In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW
import pandas as pd 
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import load_dataset

# --- Configuration ---
MODEL_NAME = "answerdotai/ModernBERT-base"
DATASET_NAME = "cardiffnlp/tweet_eval"
DATASET_CONFIG = "irony" # Specify the 'irony' subset
NUM_LABELS = 2  # Binary classification (irony vs. non-irony)
MAX_LENGTH = 128 # Max sequence length for tweets (adjust as needed)
BATCH_SIZE = 16   # Adjust based on your GPU memory
EPOCHS = 3       # Number of training epochs (Paper used 2 for SST-2 [cite: 454])
LEARNING_RATE = 5e-5 # Example learning rate (Paper used 8e-5 for SST-2 [cite: 454])
WEIGHT_DECAY = 5e-5 # Standard value for AdamW (Paper used 1e-5 for SST-2 [cite: 454])

In [2]:
# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. Load Tokenizer ---
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- 2. Load and Prepare Dataset ---
print(f"Loading '{DATASET_CONFIG}' subset of '{DATASET_NAME}' dataset...")
ds = load_dataset(DATASET_NAME, DATASET_CONFIG)

# The dataset structure is typically:
# DatasetDict({
#     train: Dataset({
#         features: ['text', 'label'],
#         num_rows: ...
#     })
#     test: Dataset({...})
#     validation: Dataset({...})
# })
# We'll use 'text' as input and 'label' as the target.

# As per your snippet, if you want to convert to pandas DataFrames first:
# train_df = ds['train'].to_pandas()
# val_df = ds['validation'].to_pandas()
# test_df = ds['test'].to_pandas()
# print(f"Train records: {len(train_df)}, Validation records: {len(val_df)}, Test records: {len(test_df)}")
# For this script, we'll directly use the Hugging Face Dataset objects for efficiency.

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

print("Tokenizing dataset...")
tokenized_ds = ds.map(tokenize_function, batched=True)

# Remove original text column to avoid issues with default collator
tokenized_ds = tokenized_ds.remove_columns(["text"])
# Rename 'label' to 'labels' as expected by the model
tokenized_ds = tokenized_ds.rename_column("label", "labels")
# Set format for PyTorch
tokenized_ds.set_format("torch")

train_dataset = tokenized_ds["train"]
val_dataset = tokenized_ds["validation"]
test_dataset = tokenized_ds["test"] # Keep test set for final evaluation

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Number of training batches: {len(train_dataloader)}")
print(f"Number of validation batches: {len(val_dataloader)}")

# --- 3. Load Model ---
print(f"Loading model {MODEL_NAME} for sequence classification...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.to(device)

# --- 4. Optimizer and Scheduler ---
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

num_training_steps = EPOCHS * len(train_dataloader)
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0, # You can set a warmup, e.g., int(0.1 * num_training_steps)
    num_training_steps=num_training_steps
)

# --- 5. Training and Evaluation Loop ---
def compute_metrics(preds, labels):
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

print("Starting training...")
for epoch in range(EPOCHS):
    model.train() # Set model to training mode
    total_train_loss = 0
    for batch_idx, batch in enumerate(train_dataloader):
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        if (batch_idx + 1) % (len(train_dataloader) // 10) == 0: # Print progress roughly 10 times per epoch
             if len(train_dataloader) > 10 : # Avoid division by zero for very small datasets
                print(f"  Epoch {epoch + 1}/{EPOCHS}, Batch {batch_idx + 1}/{len(train_dataloader)}, Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1}/{EPOCHS} finished. Average Training Loss: {avg_train_loss:.4f}")

    # --- Validation Step ---
    model.eval() # Set model to evaluation mode
    total_eval_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad(): # Disable gradient calculations
        for batch in val_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_eval_loss += loss.item()

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_eval_loss / len(val_dataloader)
    val_metrics = compute_metrics(all_preds, all_labels)
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}, F1: {val_metrics['f1']:.4f}")

print("Training complete!")

# --- 6. Final Evaluation on Test Set ---
print("\nEvaluating on the test set...")
model.eval()
total_test_loss = 0
all_test_preds = []
all_test_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_test_loss += loss.item()

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        all_test_preds.extend(predictions.cpu().numpy())
        all_test_labels.extend(labels.cpu().numpy())

avg_test_loss = total_test_loss / len(test_dataloader)
test_metrics = compute_metrics(all_test_preds, all_test_labels)
print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test F1 Score: {test_metrics['f1']:.4f}")
print(f"Test Precision: {test_metrics['precision']:.4f}")
print(f"Test Recall: {test_metrics['recall']:.4f}")

Using device: cuda
Loading tokenizer for answerdotai/ModernBERT-base...
Loading 'irony' subset of 'cardiffnlp/tweet_eval' dataset...
Tokenizing dataset...
Number of training batches: 179
Number of validation batches: 60
Loading model answerdotai/ModernBERT-base for sequence classification...


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training...
  Epoch 1/3, Batch 17/179, Loss: 0.6822
  Epoch 1/3, Batch 34/179, Loss: 0.7657


KeyboardInterrupt: 